In [ ]:
%pip install -q -U "crewai[google-genai,tools]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.

In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Please add GEMINI_API_KEY to Google Colab Secrets.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

print("Gemini configured successfully!")

Gemini configured successfully!


In [ ]:
import pandas as pd
from io import StringIO

csv_input = input(
    "Paste your sales data as CSV (include the column headers):\n"
).strip()

if not csv_input:
    print("Please enter valid sales data.")
else:
    try:
        sales_df = pd.read_csv(StringIO(csv_input))

        if sales_df.empty or len(sales_df.columns) < 2:
            print("Please provide valid CSV data with column headers.")
        else:
            sales_data = sales_df.to_dict(orient="list")

            result = await data_report_crew.kickoff_async(
                inputs={"sales_data": sales_data}
            )

            print("\nData Report Crew completed!")
            print(result)

    except Exception as e:
        print("Could not read the CSV data:", e)

,Date,Product,Region,Units_Sold,Unit_Price,Revenue
0,2026-10-01,Laptop,South,5,55000,275000
1,2026-10-02,Phone,North,12,25000,300000
2,2026-10-03,Headphones,East,20,2000,40000
3,2026-10-04,Laptop,West,4,55000,220000
4,2026-10-05,Tablet,South,8,30000,240000
5,2026-10-06,Phone,North,10,25000,250000
6,2026-10-07,Headphones,East,25,2000,50000
7,2026-10-08,Tablet,West,7,30000,210000
8,2026-10-09,Laptop,South,6,55000,330000
9,2026-10-10,Phone,North,15,25000,375000


sales.csv created successfully!


In [ ]:
from crewai_tools import FileReadTool, FileWriterTool

file_reader = FileReadTool()
file_writer = FileWriterTool()

print("File tools initialized successfully!")

File tools initialized successfully!


In [ ]:
class SalesReport(BaseModel):
    total_records: int = Field(description="Number of sales records")
    total_units_sold: int = Field(description="Total units sold")
    total_revenue: float = Field(description="Total revenue across all records")
    best_selling_product: str = Field(description="Product with the highest total units sold")
    highest_revenue_region: str = Field(description="Region with the highest total revenue")
    average_revenue_per_record: float = Field(description="Average revenue per sales record")

In [ ]:
data_analyst = Agent(
    role="Sales Data Analyst",
    goal="Analyze sales data accurately and produce structured sales statistics.",
    backstory=(
        "You are a careful data analyst who examines sales records, "
        "calculates business metrics, and identifies product and regional trends. "
        "Use the supplied CSV data and never invent figures."
    ),
    llm=llm,
    tools=[file_reader],
    verbose=True
)

In [ ]:

analysis_task = Task(
    description="""
    Analyze the sales data stored in sales.csv.

    Calculate the following metrics:
    1. Total number of sales records.
    2. Total units sold across all products.
    3. Total revenue across all records.
    4. Best-selling product based on total units sold.
    5. Region with the highest total revenue.
    6. Average revenue per sales record.

    Instructions:
    - Read the CSV file using the available file-reading tool.
    - Use the actual data to calculate every metric.
    - Group records by product to identify the best-selling product.
    - Group records by region to identify the highest-revenue region.
    - Do not guess or invent values.
    - Return the results in the required structured format.
    """,
    expected_output=(
        "A structured sales analysis containing all six metrics, "
        "with accurate values derived from sales.csv."
    ),
    agent=data_analyst,
    output_pydantic=SalesReport
)
print("Analysis task created successfully!")


Analysis task created successfully!


In [ ]:

insight_writer = Agent(
    role="Business Insight Writer",
    goal=(
        "Convert sales statistics into clear, useful business "
        "insights and recommendations."
    ),
    backstory=(
        "You are a business analyst who explains sales trends "
        "in simple language and provides practical recommendations "
        "based only on the available data."
    ),
    llm=llm,
    verbose=True
)

print("Insight Writer agent created successfully!")


Insight Writer agent created successfully!


In [ ]:

reviewer = Agent(
    role="Sales Report Reviewer",
    goal=(
        "Review the sales report for numerical accuracy, "
        "clarity, consistency, and actionable recommendations."
    ),
    backstory=(
        "You are a meticulous report reviewer. You verify claims "
        "against the supplied analysis, flag unsupported conclusions, "
        "and improve the report's readability."
    ),
    llm=llm,
    verbose=True
)

print("Reviewer agent created successfully!")


Reviewer agent created successfully!


In [ ]:

insight_task = Task(
    description="""
    Using the sales analysis provided in the task context:

    1. Explain the overall sales performance.
    2. Identify the strongest-selling product.
    3. Explain which region generated the most revenue.
    4. Discuss what the average revenue per record indicates.
    5. Provide three practical business recommendations.

    Base every numerical claim on the analysis.
    Do not invent trends or claim growth over time because the
    dataset covers only a small sample of records.
    Write the report in clear Markdown with headings and bullet points.
    """,
    expected_output=(
        "A Markdown sales report containing a summary, key findings, "
        "and three data-supported business recommendations."
    ),
    agent=insight_writer,
    context=[analysis_task]
)

print("Insight writing task created successfully!")


Insight writing task created successfully!


In [ ]:

review_task = Task(
    description="""
    Review the draft sales report using the original structured
    sales analysis available in the task context.

    Check:
    1. Whether numerical claims match the analysis.
    2. Whether product and regional findings are consistent.
    3. Whether recommendations follow from the available evidence.
    4. Whether the report is clear, well-organized, and professional.

    Correct errors and return the complete revised report in Markdown.
    Do not invent additional statistics.
    """,
    expected_output=(
        "A reviewed, corrected, professional Markdown sales report."
    ),
    agent=reviewer,
    context=[analysis_task, insight_task],
    output_file="report.md"
)

print("Review task created successfully!")


Review task created successfully!


In [ ]:

sales_crew = Crew(
    agents=[
        data_analyst,
        insight_writer,
        reviewer
    ],
    tasks=[
        analysis_task,
        insight_task,
        review_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Data Report Crew assembled successfully!")


Data Report Crew assembled successfully!


In [ ]:

result = await sales_crew.kickoff_async()

print("\n===== FINAL SALES REPORT =====\n")
print(result)

print("\nCheck the Colab Files panel for report.md")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: becc2357-827f-45c0-8c3e-a6a72550c79f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analyze the sales data stored in sales.csv.                                                                │
│                                                                                                                 │
│      Calculate the following metrics:                                                                           │
│      1. Total number of sales records.                                                                          │
│      2. Total units sold across all products.                                                                   │
│      3. Total revenue across all records.                                                                       │
│      4. Best-selling product based on total units sold.                                                         │
│      5. Region with the highest total revenue.                                                                  │
│      6. Average revenue per sales record.                                                                       │
│                                                                                                                 │
│      Instructions:                                                                                              │
│      - Read the CSV file using the available file-reading tool.                                                 │
│      - Use the actual data to calculate every metric.                                                           │
│      - Group records by product to identify the best-selling product.                                           │
│      - Group records by region to identify the highest-revenue region.                                          │
│      - Do not guess or invent values.                                                                           │
│      - Return the results in the required structured format.                                                    │
│                                                                                                                 │
│  ID: e1a5096a-9b82-496f-a851-3a81d0fe86c7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sales Data Analyst                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analyze the sales data stored in sales.csv.                                                                │
│                                                                                                                 │
│      Calculate the following metrics:                                                                           │
│      1. Total number of sales records.                                                                          │
│      2. Total units sold across all products.                                                                   │
│      3. Total revenue across all records.                                                                       │
│      4. Best-selling product based on total units sold.                                                         │
│      5. Region with the highest total revenue.                                                                  │
│      6. Average revenue per sales record.                                                                       │
│                                                                                                                 │
│      Instructions:                                                                                              │
│      - Read the CSV file using the available file-reading tool.                                                 │
│      - Use the actual data to calculate every metric.                                                           │
│      - Group records by product to identify the best-selling product.                                           │
│      - Group records by region to identify the highest-revenue region.                                          │
│      - Do not guess or invent values.                                                                           │
│      - Return the results in the required structured format.                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'file_path': 'sales.csv', 'start_line': 1, 'line_count': None}                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_a_files_content executed with result: Date,Product,Region,Units_Sold,Unit_Price,Revenue
2026-10-01,Laptop,South,5,55000,275000
2026-10-02,Phone,North,12,25000,300000
2026-10-03,Headphones,East,20,2000,40000
2026-10-04,Laptop,West,4,55000,...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sales Report Reviewer                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Sales Performance Report                                                                                     │
│                                                                                                                 │
│  ## 1. Overall Sales Performance                                                                                │
│  * **Total Records:** 10                                                                                        │
│  * **Total Units Sold:** 112                                                                                    │
│  * **Total Revenue:** ₹2,290,000                                                                                │
│  * **Average Revenue per Record:** ₹229,000.00                                                                  │
│                                                                                                                 │
│  ## 2. Product Performance                                                                                      │
│  * **Best-Selling Product (by Units Sold):** Headphones, with a total of 45 units sold.                         │
│  * **Highest Revenue Product:** Phone. *(Note: While Headphones lead in volume with 45 units sold, Phone        │
│  generated the highest total revenue).*                                                                         │
│                                                                                                                 │
│  ## 3. Regional Performance                                                                                     │
│  * **Highest Revenue Region:** North region.                                                                    │
│  * **Verified Regional Revenue:** ₹925,000.                                                                     │
│                                                                                                                 │
│  ## 4. Business Recommendations                                                                                 │
│  1. **Capitalize on High-Volume Demand for Headphones:** Since Headphones are the best-selling product by       │
│  units sold (45 units), ensure sufficient inventory and marketing support are maintained for this item to       │
│  sustain steady transaction volumes.                                                                            │
│  2. **Focus Revenue Strategy on Phones:** Given that the Phone generates the highest revenue, analyze its       │
│  pricing and bundling strategies to maximize profitability across all sales records.                            │
│  3. **Target Growth in the North Region:** With the North region delivering the highest revenue (₹925,000),     │
│  consider allocating additional sales and marketing resources to this area to maintain its strong performance.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the draft sales report against these verified metrics:                                                  │
│                                                                                                                 │
│  {'total_records': 10, 'total_units_sold': 112, 'total_revenue': 2290000, 'best_selling_product':               │
│  'Headphones', 'best_selling_product_units': 45, 'highest_revenue_product': 'Phone', 'highest_revenue_region':  │
│  'North', 'highest_revenue_region_total': 925000, 'average_revenue_per_record': 229000.0}                       │
│                                                                                                                 │
│  Check that:                                                                                                    │
│  1. Total records, units sold, total revenue, and average revenue                                               │
│     match the verified values.                                                                                  │
│  2. The best-selling product by units is correctly identified.                                                  │
│  3. The highest-revenue product is correctly identified.                                                        │
│  4. The highest-revenue region and its revenue are correct.                                                     │
│  5. Every numerical claim is supported by the verified metrics.                                                 │
│  6. Recommendations are reasonable and do not claim unsupported                                                 │
│     sales growth or customer behavior.                                                                          │
│                                                                                                                 │
│  Correct every discrepancy and return the complete final report                                                 │
│  in Markdown.                                                                                                   │
│                                                                                                                 │
│  Use ₹ for currency. Do not invent figures or alter verified values.                                            │
│                                                                                                                 │
│  Agent: Sales Report Reviewer                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL SALES REPORT =====

# Sales Performance Report

## 1. Overall Sales Performance
* **Total Records:** 10
* **Total Units Sold:** 112
* **Total Revenue:** ₹2,290,000
* **Average Revenue per Record:** ₹229,000.00

## 2. Product Performance
* **Best-Selling Product (by Units Sold):** Headphones, with a total of 45 units sold.
* **Highest Revenue Product:** Phone. *(Note: While Headphones lead in volume with 45 units sold, Phone generated the highest total revenue).*

## 3. Regional Performance
* **Highest Revenue Region:** North region.
* **Verified Regional Revenue:** ₹925,000.

## 4. Business Recommendations
1. **Capitalize on High-Volume Demand for Headphones:** Since Headphones are the best-selling product by units sold (45 units), ensure sufficient inventory and marketing support are maintained for this item to sustain steady transaction volumes.
2. **Focus Revenue Strategy on Phones:** Given that the Phone generates the highest revenue, analyze its pricing and bundling s

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: becc2357-827f-45c0-8c3e-a6a72550c79f                                                                       │
│  Final Output: # Sales Performance Report                                                                       │
│                                                                                                                 │
│  ## 1. Overall Sales Performance                                                                                │
│  * **Total Records:** 10                                                                                        │
│  * **Total Units Sold:** 112                                                                                    │
│  * **Total Revenue:** ₹2,290,000                                                                                │
│  * **Average Revenue per Record:** ₹229,000.00                                                                  │
│                                                                                                                 │
│  ## 2. Product Performance                                                                                      │
│  * **Best-Selling Product (by Units Sold):** Headphones, with a total of 45 units sold.                         │
│  * **Highest Revenue Product:** Phone. *(Note: While Headphones lead in volume with 45 units sold, Phone        │
│  generated the highest total revenue).*                                                                         │
│                                                                                                                 │
│  ## 3. Regional Performance                                                                                     │
│  * **Highest Revenue Region:** North region.                                                                    │
│  * **Verified Regional Revenue:** ₹925,000.                                                                     │
│                                                                                                                 │
│  ## 4. Business Recommendations                                                                                 │
│  1. **Capitalize on High-Volume Demand for Headphones:** Since Headphones are the best-selling product by       │
│  units sold (45 units), ensure sufficient inventory and marketing support are maintained for this item to       │
│  sustain steady transaction volumes.                                                                            │
│  2. **Focus Revenue Strategy on Phones:** Given that the Phone generates the highest revenue, analyze its       │
│  pricing and bundling strategies to maximize profitability across all sales records.                            │
│  3. **Target Growth in the North Region:** With the North region delivering the highest revenue (₹925,000),     │
│  consider allocating additional sales and marketing resources to this area to maintain its strong performance.  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:

verified_metrics = {
    "total_records": int(len(df)),
    "total_units_sold": int(df["Units_Sold"].sum()),
    "total_revenue": int(df["Revenue"].sum()),
    "best_selling_product": str(
        df.groupby("Product")["Units_Sold"].sum().idxmax()
    ),
    "best_selling_product_units": int(
        df.groupby("Product")["Units_Sold"].sum().max()
    ),
    "highest_revenue_product": str(
        df.groupby("Product")["Revenue"].sum().idxmax()
    ),
    "highest_revenue_region": str(
        df.groupby("Region")["Revenue"].sum().idxmax()
    ),
    "highest_revenue_region_total": int(
        df.groupby("Region")["Revenue"].sum().max()
    ),
    "average_revenue_per_record": float(
        df["Revenue"].mean()
    )
}

print(verified_metrics)


{'total_records': 10, 'total_units_sold': 112, 'total_revenue': 2290000, 'best_selling_product': 'Headphones', 'best_selling_product_units': 45, 'highest_revenue_product': 'Phone', 'highest_revenue_region': 'North', 'highest_revenue_region_total': 925000, 'average_revenue_per_record': 229000.0}


In [ ]:

insight_task.description = f"""
Write a professional sales performance report using these
verified metrics calculated directly from sales.csv:

{verified_metrics}

Include the following sections:

1. Overall Sales Performance
   - Total records, units sold, total revenue, and average
     revenue per record.

2. Product Performance
   - Identify the best-selling product by units sold.
   - Identify the product generating the highest revenue.
   - Clearly distinguish units sold from revenue generated.

3. Regional Performance
   - Identify the region with the highest revenue.
   - Include its verified revenue figure.

4. Business Recommendations
   - Provide three practical recommendations supported by
     the available data.

Rules:
- Use only the verified numerical values supplied above.
- Do not recalculate, modify, or invent statistics.
- Use ₹ as the currency symbol.
- Do not claim sales growth or trends over time.
- Format the report using clean Markdown headings and bullets.
"""

print("Insight Writer task updated successfully!")


Insight Writer task updated successfully!


In [ ]:

review_task.description = f"""
Review the draft sales report against these verified metrics:

{verified_metrics}

Check that:
1. Total records, units sold, total revenue, and average revenue
   match the verified values.
2. The best-selling product by units is correctly identified.
3. The highest-revenue product is correctly identified.
4. The highest-revenue region and its revenue are correct.
5. Every numerical claim is supported by the verified metrics.
6. Recommendations are reasonable and do not claim unsupported
   sales growth or customer behavior.

Correct every discrepancy and return the complete final report
in Markdown.

Use ₹ for currency. Do not invent figures or alter verified values.
"""

print("Reviewer task updated successfully!")


Reviewer task updated successfully!


In [ ]:

import os

report_path = "report.md"

if os.path.exists(report_path):
    print("Report saved successfully!")
    print("File size:", os.path.getsize(report_path), "bytes")

    with open(report_path, "r", encoding="utf-8") as file:
        print("\n===== report.md =====\n")
        print(file.read())
else:
    print("report.md was not found in the current directory.")
    print("Check the Colab Files panel.")


Report saved successfully!
File size: 1253 bytes

===== report.md =====

# Sales Performance Report

## 1. Overall Sales Performance
* **Total Records:** 10
* **Total Units Sold:** 112
* **Total Revenue:** ₹2,290,000
* **Average Revenue per Record:** ₹229,000.00

## 2. Product Performance
* **Best-Selling Product (by Units Sold):** Headphones, with a total of 45 units sold.
* **Highest Revenue Product:** Phone. *(Note: While Headphones lead in volume with 45 units sold, Phone generated the highest total revenue).*

## 3. Regional Performance
* **Highest Revenue Region:** North region.
* **Verified Regional Revenue:** ₹925,000.

## 4. Business Recommendations
1. **Capitalize on High-Volume Demand for Headphones:** Since Headphones are the best-selling product by units sold (45 units), ensure sufficient inventory and marketing support are maintained for this item to sustain steady transaction volumes.
2. **Focus Revenue Strategy on Phones:** Given that the Phone generates the highest rev